In [1]:
import pandas as pd
import wm_unit as wmu
from datetime import datetime
from datetime import timedelta

def next_turn(current_time):
    current_time += timedelta(minutes=15)
    print(current_time)
    return current_time


def create_ground_unit(
    unit_id,
    side="blue",
    inf_df=None,
    armor_df=None,
):
    """
    Универсальный конструктор ground unit

    Варианты:
    1) inf_df only       -> infantry
    2) armor_df only     -> armor
    3) inf_df + armor_df -> mech
    """

    # ---------- validation ----------
    if inf_df is None and armor_df is None:
        raise ValueError("Need inf_df and/or armor_df")

    # ---------- INF PREP ----------
    if inf_df is not None:
        inf_df = inf_df.copy().reset_index(drop=True)

        if "power" not in inf_df.columns:
            inf_df["power"] = 1

        if "inf_kills" not in inf_df.columns:
            inf_df["inf_kills"] = 0

        if "apc_kills" not in inf_df.columns:
            inf_df["apc_kills"] = 0


    # ---------- ARMOR PREP ----------
    if armor_df is not None:
        armor_df = armor_df.copy().reset_index(drop=True)

        if "power" not in armor_df.columns:
            armor_df["power"] = 8

        if "type" not in armor_df.columns:
            armor_df["type"] = "apc"

        if "transport" not in armor_df.columns:
            armor_df["transport"] = 9

        if "inf_kills" not in armor_df.columns:
            armor_df["inf_kills"] = 0

        if "apc_kills" not in armor_df.columns:
            armor_df["apc_kills"] = 0


    # ==================================================
    # 1 ARM UNIT
    # ==================================================
    if inf_df is None and armor_df is not None:

        arm = wmu.Unit(
            unit_id=unit_id,
            overall_type="arm",
            personal_type="arm",
            alive_df=armor_df,
            cas_df=pd.DataFrame(columns=armor_df.columns),
            side=side,
        )

        arm.size = "ARMOR"

        return arm


    # ==================================================
    # 2 INF UNIT
    # ==================================================
    if inf_df is not None and armor_df is None:

        inf = wmu.Unit(
            unit_id=unit_id,
            overall_type="inf",
            personal_type="inf",
            alive_df=inf_df,
            cas_df=pd.DataFrame(columns=inf_df.columns),
            side=side,
        )

        return inf


    # ==================================================
    # 3 MECH UNIT
    # ==================================================
    armor_part = wmu.Unit(
        unit_id=unit_id + "_ARM",
        overall_type="arm",
        personal_type="arm",
        alive_df=armor_df,
        cas_df=pd.DataFrame(columns=armor_df.columns),
        side=side,
    )

    armor_part.size = "ARMOR"

    mech = wmu.Unit(
        unit_id=unit_id,
        overall_type="inf",
        personal_type="inf",
        alive_df=inf_df,
        cas_df=pd.DataFrame(columns=inf_df.columns),
        side=side,
    )

    mech.armor_part = armor_part
    mech.update_current_type()   # станет mech

    return mech






def run_ground_battle(
    attacker,
    defender,
    current_time=None,
    attacker_cover=0,
    defender_cover=0,
    attacker_elevation=0,
    defender_elevation=0,
    distance=0,
    defender_returns_fire=True,
    attacker_berserk=False,
    defender_berserk=False,
    armor_is_moving=False,
    armor_is_far=False,
):
    """
    Запускает ОДИН ground battle между двумя Unit.

    defender_returns_fire=True  -> обоюдный бой
    defender_returns_fire=False -> односторонняя атака без ответа
    """

    global ground_logs

    if current_time is None:
        current_time = datetime.now()

    context = BattleContext(
        attacker_cover=attacker_cover,
        defender_cover=defender_cover,
        attacker_elevation=attacker_elevation,
        defender_elevation=defender_elevation,
        distance=distance,
        defender_returns_fire=defender_returns_fire,
        attacker_berserk=attacker_berserk,
        defender_berserk=defender_berserk,
        armor_is_moving=armor_is_moving,
        armor_is_far=armor_is_far,
    )

    result = engine.resolve_engagement(
        attacker=attacker,
        defender=defender,
        context=context,
        logs=ground_logs,
        current_time=current_time,
    )

    ground_logs = result.logs

    return result


from datetime import datetime
import pandas as pd
from ground_engine import GroundEngine, BattleContext

engine = GroundEngine()

GROUND_LOG_COLUMNS = [
    "current_time",
    "initiator",
    "log_blue_id",
    "log_blue_type",
    "log_blue_inf_force",
    "log_blue_arm_force",
    "log_blue_cas_inf",
    "log_blue_cas_armor",
    "log_red_id",
    "log_red_type",
    "log_red_inf_force",
    "log_red_arm_force",
    "log_red_cas_inf",
    "log_red_cas_armor",
    "log_attack_type",
    "log_result",
]

ground_logs = pd.DataFrame(columns=GROUND_LOG_COLUMNS)


In [2]:

from force_manager import ForceManager
from ground_engine import GroundEngine, BattleContext
import wm_unit as wmu

# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------
PLAYER_FILE = "player_brigade.xlsx"
ENEMY_FILE = "enemy_brigade.xlsx"

player_manager = ForceManager.from_excel(PLAYER_FILE)
enemy_manager = ForceManager.from_excel(ENEMY_FILE)



In [4]:
apc = player_manager.armor_df.iloc[:6]

In [5]:
pc2 = 0
pc3 = 0

pp = [[pc2,['B2-C2'],'company_uid','pc2'],
[pc3,['B1-C3'],'company_uid','pc3']]

counter = 0
for p in pp:
    inf_df = player_manager.master_df[(player_manager.master_df[p[2]].isin(p[1]))&
                                     (player_manager.master_df['status']=='alive')]

    pp[counter][0] = create_ground_unit(
        unit_id=p[3],
        side="blue",
        inf_df=inf_df)
    counter+=1
    
pc2 = pp[0][0]
pc3 = pp[1][0]



In [7]:
pc3

Unit(unit_id=pc3, overall_type=inf,personal_type=inf,size=CM, alive_df=126, cas_df=0, armor_part=0, last_parameters={'elev_pos': 0, 'cover_level': 0, 'target_distance': 0, 'enemy_unit_id': 'id1', 'berserk_mode': 0})

In [31]:
inf_df = enemy_manager.master_df[
    enemy_manager.master_df['platoon_uid'].isin(['B9-C2-P3'])&
    (enemy_manager.master_df['status']=='alive')]

ec2p3  = create_ground_unit(
    unit_id='ec2p3',
    side="red",
    inf_df=inf_df)


In [9]:
START_TIME = datetime(2026, 4, 6, 6, 0)

In [34]:
START_TIME = next_turn(START_TIME)

2026-04-06 07:30:00


In [23]:
ec2p1

Unit(unit_id=ec2p1, overall_type=inf,personal_type=inf,size=PT, alive_df=42, cas_df=0, armor_part=0, last_parameters={'elev_pos': 0, 'cover_level': 0, 'target_distance': 0, 'enemy_unit_id': 'id1', 'berserk_mode': 0})

In [32]:
battle_result = run_ground_battle(
    attacker=ec2p3,
    defender=pc2,
    current_time=str(START_TIME),
    attacker_cover=1,
    defender_cover=1,
    attacker_elevation=0,
    defender_elevation=0,
    distance=1,
    defender_returns_fire=False,   # False = атака без ответа
)



In [ ]:
battle_result = run_ground_battle(
    attacker=p,
    defender=ec2p3,
    current_time=str(START_TIME),
    attacker_cover=1,
    defender_cover=2,
    attacker_elevation=0,
    defender_elevation=0,
    distance=1,
    defender_returns_fire=False,   # False = атака без ответа
)



In [33]:
battle_result.logs

,current_time,initiator,log_blue_id,log_blue_type,log_blue_inf_force,log_blue_arm_force,log_blue_cas_inf,log_blue_cas_armor,log_red_id,log_red_type,log_red_inf_force,log_red_arm_force,log_red_cas_inf,log_red_cas_armor,log_attack_type,log_result
0,2026-04-06 06:15:00,blue,pc3,inf,126,0,2,0,ec1,inf,126,0,21,0,inf_attacks_inf,победа
1,2026-04-06 06:45:00,blue,pc3,inf,124,0,1,0,ec2p1,inf,42,0,21,0,inf_attacks_inf,победа
2,2026-04-06 07:15:00,blue,pc2,inf,76,0,2,0,ec2p2,inf,42,0,0,0,inf_attacks_inf,ничья
3,2026-04-06 07:15:00,red,pc2,inf,74,0,11,0,ec2p3,inf,42,0,0,0,inf_attacks_inf,side_attack


In [83]:
for podr in [pc1,ppp,pp3]:
    cas = podr.cas_df['global_soldier_id']
    podr.cas_df['status'] = 'killed'
    player_manager.master_df = player_manager.master_df[~player_manager.master_df['global_soldier_id'].isin(cas)]
    player_manager.master_df = player_manager.master_df.append(podr.cas_df)
    
    alive = podr.alive_df['global_soldier_id']
    player_manager.master_df = player_manager.master_df[~player_manager.master_df['global_soldier_id'].isin(alive)]
    player_manager.master_df = player_manager.master_df.append(podr.alive_df)

<ipython-input-83-5b54000122be>:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  player_manager.master_df = player_manager.master_df.append(podr.cas_df)
<ipython-input-83-5b54000122be>:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  player_manager.master_df = player_manager.master_df.append(podr.alive_df)
<ipython-input-83-5b54000122be>:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  player_manager.master_df = player_manager.master_df.append(podr.cas_df)
<ipython-input-83-5b54000122be>:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  player_manager.master_df = player_manager.master_df.append(podr.alive_df)
<ipython-input-83-5b54000122

In [81]:
for podr in [ep1,ec1,ec2]:
    cas = podr.cas_df['global_soldier_id']
    podr.cas_df['status'] = 'killed'
    enemy_manager.master_df = enemy_manager.master_df[~enemy_manager.master_df['global_soldier_id'].isin(cas)]
    enemy_manager.master_df = enemy_manager.master_df.append(podr.cas_df)
    
    alive = podr.alive_df['global_soldier_id']
    enemy_manager.master_df = enemy_manager.master_df[~enemy_manager.master_df['global_soldier_id'].isin(alive)]
    enemy_manager.master_df = enemy_manager.master_df.append(podr.alive_df)

<ipython-input-81-a414a6b7e3fa>:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  enemy_manager.master_df = enemy_manager.master_df.append(podr.cas_df)
<ipython-input-81-a414a6b7e3fa>:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  enemy_manager.master_df = enemy_manager.master_df.append(podr.alive_df)
<ipython-input-81-a414a6b7e3fa>:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  enemy_manager.master_df = enemy_manager.master_df.append(podr.cas_df)
<ipython-input-81-a414a6b7e3fa>:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  enemy_manager.master_df = enemy_manager.master_df.append(podr.alive_df)
<ipython-input-81-a414a6b7e3fa>:5: F

In [10]:
podr = pd.read_excel('saved\\pppcas.xlsx')

In [11]:
cas = podr['global_soldier_id']
podr['status'] = 'killed'
player_manager.master_df = player_manager.master_df[~player_manager.master_df['global_soldier_id'].isin(cas)]
player_manager.master_df = player_manager.master_df.append(podr)

<ipython-input-11-a344bd74dd49>:4: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  player_manager.master_df = player_manager.master_df.append(podr)


In [12]:
player_manager.save_to_excel(PLAYER_FILE)
enemy_manager.save_to_excel(ENEMY_FILE)

In [51]:
ground_logs.to_csv('saved\\0 LOGS-d1-b2.xlsx',index=False)